# Notebook 01 — Price Scraping
**Owner:** Person 1

**Goal:** Scrape hardware prices from PCPartPicker / Newegg and build a clean weekly price time series.

**Output:** `../data/prices/prices_clean.csv`

**Columns:** `date | product | category | price_usd | retailer | url`

In [ ]:
import requests
from bs4 import BeautifulSoup
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
import pandas as pd
import time
import re
from datetime import datetime

## 1. Define Target Products
Start with a small set and expand once pipeline works.

In [ ]:
# PCPartPicker product URLs — update these as needed
TARGET_PRODUCTS = [
    {
        "name": "Corsair Vengeance DDR5-6000 32GB",
        "category": "RAM",
        "url": "https://pcpartpicker.com/product/search/?q=corsair+vengeance+ddr5+6000"
    },
    {
        "name": "G.Skill Trident Z5 DDR5-6000 32GB",
        "category": "RAM",
        "url": "https://pcpartpicker.com/product/search/?q=gskill+trident+z5+ddr5"
    },
    {
        "name": "NVIDIA RTX 4060",
        "category": "GPU",
        "url": "https://pcpartpicker.com/product/search/?q=rtx+4060"
    },
]
print(f"Tracking {len(TARGET_PRODUCTS)} products")

## 2. Scrape Current Prices (BeautifulSoup)
PCPartPicker search results are mostly static HTML — try requests first before falling back to Selenium.

In [ ]:
HEADERS = {
    "User-Agent": "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36"
}

def scrape_pcpartpicker_price(product: dict) -> dict:
    """Scrape current best price for a product from PCPartPicker search."""
    response = requests.get(product["url"], headers=HEADERS)
    soup = BeautifulSoup(response.text, "html.parser")
    
    # TODO: inspect PCPartPicker HTML and update selector below
    # Typical pattern: price is in a <span> or <td> with class containing 'price'
    price_tag = soup.find("span", class_=re.compile("price", re.I))
    
    price = None
    if price_tag:
        price_text = price_tag.get_text(strip=True)
        match = re.search(r'[\d,]+\.\d{2}', price_text)
        if match:
            price = float(match.group().replace(',', ''))
    
    return {
        "date": datetime.today().strftime("%Y-%m-%d"),
        "product": product["name"],
        "category": product["category"],
        "price_usd": price,
        "retailer": "PCPartPicker",
        "url": product["url"]
    }

# Test on first product
result = scrape_pcpartpicker_price(TARGET_PRODUCTS[0])
print(result)

## 3. Scrape All Products + Save Raw

In [ ]:
records = []
for product in TARGET_PRODUCTS:
    record = scrape_pcpartpicker_price(product)
    records.append(record)
    print(f"  {record['product']}: ${record['price_usd']}")
    time.sleep(1)  # be polite

df_raw = pd.DataFrame(records)
df_raw.head()

## 4. (Optional) Selenium for JavaScript-Rendered Pages
Use this if BeautifulSoup returns empty prices — some retailers load prices via JS.

In [ ]:
# Uncomment and adapt if needed

# options = webdriver.ChromeOptions()
# options.add_argument('--headless')
# driver = webdriver.Chrome(options=options)

# def scrape_with_selenium(url: str) -> str:
#     driver.get(url)
#     WebDriverWait(driver, 10).until(
#         EC.presence_of_element_located((By.CLASS_NAME, 'price'))
#     )
#     return driver.page_source

# driver.quit()

## 5. Clean and Build Weekly Time Series
Run this notebook on different days (or pull historical data) to build up the time series.

In [ ]:
import os

OUTPUT_PATH = "../data/prices/prices_clean.csv"

df_raw["date"] = pd.to_datetime(df_raw["date"])
df_raw["week"] = df_raw["date"].dt.to_period("W")

# Append to existing file if it exists
if os.path.exists(OUTPUT_PATH):
    df_existing = pd.read_csv(OUTPUT_PATH, parse_dates=["date"])
    df_out = pd.concat([df_existing, df_raw]).drop_duplicates(subset=["date", "product"])
else:
    df_out = df_raw

df_out.to_csv(OUTPUT_PATH, index=False)
print(f"Saved {len(df_out)} records to {OUTPUT_PATH}")
df_out.tail()